In [1]:
import pandas as pd
import numpy as np
import time
import math
import datetime
import seaborn as sns
import matplotlib.pylab as plt
import sklearn

import dask.dataframe as dd # To work with large data sets
from sklearn.preprocessing import StandardScaler

In [2]:
# Some globals
PROJECT_PATH  = "." # '/content/drive/MyDrive/CENG574_PROJECT'
PROJECT_RESULT_PATH = PROJECT_PATH + '/result'

FEATURE_SELECTION_DATA_PATH = PROJECT_RESULT_PATH + '/pruned_yellow_tripdata_2015-01.parquet' # Our selected features (before normalization)
NORMALIZED_DATA_PATH = PROJECT_RESULT_PATH + '/normalized_pruned_yellow_tripdata_2015-01.parquet' # Our selected features (after normalization)


In [3]:
#from google.colab import drive
#drive.mount('/content/drive')

In [4]:

df =  pd.read_parquet(FEATURE_SELECTION_DATA_PATH)
print(f"[INFO] Loaded data set with total samples: {len(df)}")

[INFO] Loaded data set with total samples: 12371288


In [5]:

df_dropna = df.dropna() # Only drops NaN, Inf remains

# stats:
print(f"Number of samples in the original dataset: {df.shape[0]}")
print(f"Number of samples with all valid measurements: {df_dropna.shape[0]}")
print(f"Number of samples with missing/invalid values overall: {df.shape[0] - df_dropna.shape[0]}")
print(f"Number of features: {df.shape[1]}")

Number of samples in the original dataset: 12371288
Number of samples with all valid measurements: 12371288
Number of samples with missing/invalid values overall: 0
Number of features: 10


In [6]:
# To inspect -Inf, Inf, NaN
# because otherwise normalization fails
bad_rows = df[
    df.isna().any(axis=1) |
    df.isin([np.inf, -np.inf]).any(axis=1)
]

print(bad_rows.head())
print("Number of bad rows (missing ):", len(bad_rows))

Empty DataFrame
Columns: [passenger_count, trip_distance, pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude, total_amount, trip_times, pickup_times, Speed]
Index: []
Number of bad rows (missing ): 0


In [7]:
df = df.drop(bad_rows.index)
print("Number of samples after dropping inf and nans:", len(df))

Number of samples after dropping inf and nans: 12371288


In [8]:
# Below is for sanity check, to make sure all of our features are numeric

def to_numpy_numeric(df):
    df = df.select_dtypes(include=[np.number])
    return df.to_numpy(), df.columns

X, feature_names = to_numpy_numeric(df)
print(X.shape)

(12371288, 10)


In [9]:
# Below is for sanity check, in our project no constant found for the selected features

def remove_constant_feats(dataset: np.ndarray, epsilon=1e-12) -> np.ndarray:
    # Check if there is 0 std in a feature,
    # this is done to avoid division by zero during nornmalization
    constant_data_idxs = dataset.std(axis=0) < epsilon

    if np.any(constant_data_idxs):
        idxs = np.where(constant_data_idxs == True)
        print("[WARNING] Found zero std in the following features at indices: ", idxs, " discarding these features...")
        dataset = dataset[:,~constant_data_idxs] # keep positive std
        print(f"[INFO] Dataset now has shape {dataset.shape}")

    else:
      print("[INFO] No constant (std = 0) feature found. Returning the same dataset...")

    return dataset

X = remove_constant_feats(X)

[INFO] No constant (std = 0) feature found. Returning the same dataset...


## Normalize Data Set


In [10]:
def assert_normalized(dataset: np.ndarray):
    assert np.allclose(dataset.std(axis=0), 1.0), f"Std check failed for: {dataset.std(axis=0)}"
    assert np.allclose(dataset.mean(axis=0), 0.0), f"Mean check failed for: {dataset.mean(axis=0)}"
    print(f"[INFO] Asserted that the dataset is normalized. Dataset mean {dataset.mean():.4f}, std {dataset.std():.4f}")

scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
assert_normalized(X_normalized) # For sanity check

[INFO] Asserted that the dataset is normalized. Dataset mean 0.0000, std 1.0000


## Save


In [11]:
df_norm = pd.DataFrame(X_normalized, columns=df.columns)

df_norm.to_parquet(
    NORMALIZED_DATA_PATH,
    index=False
)